<a href="https://colab.research.google.com/github/Anaaaslagi/Tugas1A_AnasGhifari_5026221155/blob/main/Week2_PBA_Scrapping_(Anas).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Scrapping Transjakarta Mobile

In [1]:
!pip install google_play_scraper
!pip install textblob
!pip install seaborn

In [2]:
from google_play_scraper import app
import pandas as pd
import numpy as np
import sklearn
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as dates
import seaborn as sns
import textblob
#from wordcloud import WordCloud
from pathlib import Path
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix,classification_report, accuracy_score

import pickle
import re
import time
import datetime                              # access to %%time, for timing individual notebook cells
import os
from PIL import Image
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

%matplotlib inline
%config InlineBackend.figure_format='retina'

# Import seaborn styles explicitly
import seaborn as sns
# Apply the seaborn style before creating plots
sns.set_style("whitegrid")  # This line sets the Seaborn style

plt.rcParams["figure.figsize"] = (15,10)

In [3]:
#Android App Transjakarta  from Google Play at
#https://play.google.com/store/apps/details?id=com.transjakmobile
#The apps ID found in the link after id=com.transjakmobile


from google_play_scraper import app, Sort, reviews_all

transjakarta_reviews = reviews_all(
    'com.transjakmobile',
    sleep_milliseconds=0, # defaults to 0
    lang='id', # defaults to 'en'
    sort=Sort.NEWEST, # defaults to Sort.MOST_RELEVANT
)

In [7]:
#Save Transjakarta reviews into dataframe
df_tj = pd.DataFrame(np.array(transjakarta_reviews),columns=['content'])
df_tj = df_tj.join(pd.DataFrame(df_tj.pop('content').tolist()))
df_tj.to_csv(r'df_tj_raw.csv', index=False)

In [8]:
df_tj

,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,bde4ccb0-d440-461c-8a10-ee2d6ee0e815,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,👍,5,0,2.9.0,2025-10-05 15:49:49,None,NaT,2.9.0
1,05b3aa4e-2214-4c5c-b193-d3d824f88c8d,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,aplikasi lemot gk guna.gk mau respon pas login...,1,0,2.9.0,2025-10-04 19:49:12,None,NaT,2.9.0
2,19bd7606-fa19-4908-91f4-51b8426b3434,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,baru coba,5,0,None,2025-10-04 14:06:36,None,NaT,None
3,dcbb0018-4af8-449d-80bf-c89767ec1152,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"aplikasi nya mempermudah banget, tetapi kalo o...",4,0,2.9.0,2025-10-04 13:33:11,None,NaT,2.9.0
4,a63a99d1-bc93-42b1-bd1a-d20a8170e576,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,sangat baik,5,0,None,2025-10-04 09:54:37,None,NaT,None
...,...,...,...,...,...,...,...,...,...,...,...
3011,cdd9f5f2-b4cb-44c2-814e-860e0688f3d5,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Mantaps,5,5,1.4,2024-05-16 18:10:02,None,NaT,1.4
3012,66859930-33cc-484a-9c0a-f25534958578,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Sudah enggak crash kalau pilih profil,5,7,1.8,2024-05-16 11:52:59,None,NaT,1.8
3013,eb386bcd-d074-448e-b636-8f8b4d20ecb1,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Aplikasi kocag, gk becus bikin aplikasi, masuk...",1,7,1.4,2024-05-12 16:42:40,None,NaT,1.4
3014,43b40878-dcff-44b8-bed5-2dfb89bc0880,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"Aplikasi gak jelas, server eror, loading lama,...",1,23,1.4,2024-05-11 20:43:39,"Halo Teguh Aliansyah,\nTerima kasih untuk masu...",2024-05-23 13:29:46,1.4


In [6]:
df_tj.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3016 entries, 0 to 3015
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   reviewId              3016 non-null   object        
 1   userName              3016 non-null   object        
 2   userImage             3016 non-null   object        
 3   content               3016 non-null   object        
 4   score                 3016 non-null   int64         
 5   thumbsUpCount         3016 non-null   int64         
 6   reviewCreatedVersion  2534 non-null   object        
 7   at                    3016 non-null   datetime64[ns]
 8   replyContent          4 non-null      object        
 9   repliedAt             4 non-null      datetime64[ns]
 10  appVersion            2534 non-null   object        
dtypes: datetime64[ns](2), int64(2), object(7)
memory usage: 259.3+ KB
